In [ ]:
# preprocessing_patchtst.py
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
from sklearn.preprocessing import StandardScaler
from datetime import timedelta

import torch
from torch.utils.data import Dataset

# --------------------------
# 1) Calendar feature utils
# --------------------------
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add calendar-derived features (all known at prediction time)."""
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    # Day-of-week (0~6) -> sine/cosine
    dow = df["date"].dt.dayofweek
    df["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
    df["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)

    # Month (1~12) -> sine/cosine
    month = df["date"].dt.month
    df["month_sin"] = np.sin(2 * np.pi * (month - 1) / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * (month - 1) / 12.0)

    # Weekend flag
    df["is_weekend"] = (dow >= 5).astype(int)

    return df


# -----------------------------------
# 2) Window construction (per block)
# -----------------------------------
@dataclass
class WindowMeta:
    idx: int
    store_menu: str
    input_start: pd.Timestamp
    input_end: pd.Timestamp
    label_start: pd.Timestamp
    label_end: pd.Timestamp

def build_windows(
    df: pd.DataFrame,
    lookback: int = 28,
    horizon: int = 7,
    feature_cols: Optional[List[str]] = None,
    target_col: str = "sales",
) -> Tuple[np.ndarray, np.ndarray, List[WindowMeta], List[str]]:
    """
    Build rolling windows for all store_menu blocks.
    Returns:
        X: (N, lookback, F) float32
        y: (N, horizon) float32
        metas: list of WindowMeta for each sample
        used_feature_cols: actual feature col order for X
    Notes:
        - No leakage: only past lookback steps used to predict next horizon steps.
        - Each store_menu is independent.
    """
    df = df.copy()
    if feature_cols is None:
        # sales + calendar features (order matters: sales at index 0)
        feature_cols = ["sales", "dow_sin", "dow_cos", "month_sin", "month_cos", "is_weekend"]

    X_list, y_list, metas = [], [], []

    for sm, g in df.groupby("store_menu", sort=False):
        g = g.sort_values("date")
        if len(g) < (lookback + horizon):
            continue

        feats = g[feature_cols].values.astype(np.float32)
        target = g[target_col].values.astype(np.float32)
        dates = g["date"].values

        # rolling windows
        for t in range(lookback, len(g) - horizon + 1):
            x_win = feats[t - lookback : t, :]              # [lookback, F]
            y_win = target[t : t + horizon]                 # [horizon]
            X_list.append(x_win)
            y_list.append(y_win)

            metas.append(
                WindowMeta(
                    idx=len(metas),
                    store_menu=sm,
                    input_start=pd.Timestamp(dates[t - lookback]),
                    input_end=pd.Timestamp(dates[t - 1]),
                    label_start=pd.Timestamp(dates[t]),
                    label_end=pd.Timestamp(dates[t + horizon - 1]),
                )
            )

    X = np.stack(X_list, axis=0) if X_list else np.empty((0, lookback, len(feature_cols)), dtype=np.float32)
    y = np.stack(y_list, axis=0) if y_list else np.empty((0, horizon), dtype=np.float32)
    return X, y, metas, feature_cols


# -------------------------------------------------------
# 3) Time-based K-fold with global split dates + gap
# -------------------------------------------------------
def make_time_folds(
    metas: List[WindowMeta],
    k: int,
    gap_days: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Build time-based folds using global split dates.
    - Sort unique input_end dates, split by quantiles to create k folds.
    - For fold i, define a [val_start, val_end] date range; 
      train windows are those with input_end <= (val_start - gap).
    - gap_days prevents overlap between last train inputs and first val inputs/labels.
    Returns:
        folds: list of (train_idx, val_idx)
    """
    if len(metas) == 0:
        return []

    # Use input_end as the sample's time position
    all_dates = pd.Series([m.input_end for m in metas]).sort_values().unique()
    # Choose k segments over the timeline; ensure enough points per fold
    # We'll slice by percentiles: (k segments -> k folds)
    qs = np.linspace(0, 1, k + 1)
    cut_dates = [all_dates[int(q * (len(all_dates) - 1))] for q in qs]  # inclusive boundaries

    folds = []
    meta_df = pd.DataFrame({
        "idx": [m.idx for m in metas],
        "input_end": [m.input_end for m in metas],
    })

    for i in range(k):
        val_start = cut_dates[i]
        val_end   = cut_dates[i + 1]

        # validation windows whose input_end in (val_start, val_end]
        val_mask = (meta_df["input_end"] > val_start) & (meta_df["input_end"] <= val_end)
        val_idx = meta_df.loc[val_mask, "idx"].values

        # train windows whose input_end <= (val_start - gap)
        gap_dt = val_start - pd.Timedelta(days=gap_days)
        train_mask = (meta_df["input_end"] <= gap_dt)
        train_idx = meta_df.loc[train_mask, "idx"].values

        # Skip if either side is empty (can happen on tiny datasets or large k)
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        folds.append((train_idx, val_idx))

    return folds


# --------------------------------------------
# 4) Fold-wise scaling and Dataset for PyTorch
# --------------------------------------------
class PatchTSTDataset(Dataset):
    """
    PyTorch Dataset for PatchTST.
    X: (N, L, F), y: (N, H)
    """
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.float32))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]

def fit_scaler_on_train(X_train: np.ndarray, scale_feature_indices: List[int]) -> StandardScaler:
    """
    Fit scaler on train windows only (flatten over lookback).
    We typically scale only the first feature (sales).
    """
    scaler = StandardScaler()
    # collect train values for the features to scale
    # shape: (N * lookback, len(scale_feature_indices))
    L = X_train.shape[1]
    feats = X_train[:, :, scale_feature_indices].reshape(-1, len(scale_feature_indices))
    scaler.fit(feats)
    return scaler

def apply_scaler(
    X: np.ndarray,
    y: np.ndarray,
    scaler: StandardScaler,
    x_scale_idx: List[int],
    y_scale: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Apply fitted scaler to X (selected feature columns across lookback) and y (if target corresponds to the same scaled signal, e.g., sales).
    Assumes the first x_scale_idx refers to 'sales'.
    """
    X = X.copy()
    y = y.copy()

    # scale X
    L = X.shape[1]
    feats = X[:, :, x_scale_idx].reshape(-1, len(x_scale_idx))
    feats_scaled = scaler.transform(feats)
    X[:, :, x_scale_idx] = feats_scaled.reshape(-1, L, len(x_scale_idx))

    # scale y (only the first dimension i.e., 'sales' target)
    if y_scale:
        # y is (N, horizon) of 'sales' values
        # Build dummy array with same columns to reuse scaler.mean_/scale_ on the first column
        y_flat = y.reshape(-1, 1)
        y_scaled = (y_flat - scaler.mean_[0]) / scaler.scale_[0]
        y = y_scaled.reshape(y.shape)

    return X.astype(np.float32), y.astype(np.float32)


# --------------------------
# 5) End-to-end pipeline API
# --------------------------
@dataclass
class PreprocessConfig:
    lookback: int = 28
    horizon: int = 7
    k_folds: int = 5
    gap_days: int = 7                # to avoid leakage at fold boundary
    sales_feature_first: bool = True # X[..., 0] == 'sales'

def preprocess_and_make_folds(
    train_csv: str,
    cfg: PreprocessConfig = PreprocessConfig(),
) -> Dict[str, any]:
    """
    Load train.csv, create windows, build time-based K-folds,
    and return per-fold datasets with fitted scalers.
    """
    # Load
    df = pd.read_csv(train_csv)
    df["date"] = pd.to_datetime(df["date"])
    df = add_calendar_features(df)

    # Windows
    feature_cols = ["sales", "dow_sin", "dow_cos", "month_sin", "month_cos", "is_weekend"]
    X, y, metas, used_feats = build_windows(
        df,
        lookback=cfg.lookback,
        horizon=cfg.horizon,
        feature_cols=feature_cols,
        target_col="sales",
    )

    # Folds
    folds = make_time_folds(metas, k=cfg.k_folds, gap_days=cfg.gap_days)

    # Prepare per-fold datasets
    fold_outputs = []
    x_scale_idx = [0] if cfg.sales_feature_first else []

    for fold_id, (tr_idx, va_idx) in enumerate(folds):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_va, y_va = X[va_idx], y[va_idx]

        # Fit scaler on TRAIN only
        scaler = fit_scaler_on_train(X_tr, x_scale_idx) if len(x_scale_idx) > 0 else None

        if scaler is not None:
            X_tr, y_tr = apply_scaler(X_tr, y_tr, scaler, x_scale_idx, y_scale=True)
            X_va, y_va = apply_scaler(X_va, y_va, scaler, x_scale_idx, y_scale=True)

        ds_tr = PatchTSTDataset(X_tr, y_tr)
        ds_va = PatchTSTDataset(X_va, y_va)

        fold_outputs.append(
            {
                "fold_id": fold_id,
                "train_idx": tr_idx,
                "val_idx": va_idx,
                "train_dataset": ds_tr,
                "val_dataset": ds_va,
                "scaler": scaler,
                "feature_cols": used_feats,
            }
        )

    return {
        "folds": fold_outputs,
        "metas": metas,
        "feature_cols": used_feats,
        "lookback": cfg.lookback,
        "horizon": cfg.horizon,
    }


# --------------------------
# Example usage (script)
# --------------------------
if __name__ == "__main__":
    cfg = PreprocessConfig(lookback=28, horizon=7, k_folds=5, gap_days=7)
    out = preprocess_and_make_folds("./train.csv", cfg)

    print(f"# of folds built: {len(out['folds'])}")
    for fo in out["folds"]:
        print(f"[Fold {fo['fold_id']}] train={len(fo['train_idx'])}, val={len(fo['val_idx'])}")
